In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)
print('pandas version:', pd.__version__)

# Memeriksa dan memuat dataset utama
FILE = 'peserta_pelatihan_umkm_2025.csv'

if not os.path.exists(FILE):
    try:
        from google.colab import files
        uploaded = files.upload()    # Pilih file peserta_pelatihan_umkm_2025.csv jika belum ada
    except ImportError:
        raise FileNotFoundError('/content/drive/MyDrive/eda/data/peserta_pelatihan_umkm_2025.csv')

df = pd.read_csv(FILE)
print("\n--- BERHASIL DIMUAT ---")
print(f"Ukuran Dataset: {df.shape[0]} baris x {df.shape[1]} kolom")
df.head(3)

pandas version: 2.2.3


Saving peserta_pelatihan_umkm_2025.csv to peserta_pelatihan_umkm_2025.csv

--- BERHASIL DIMUAT ---
Ukuran Dataset: 620 baris x 28 kolom


,id_peserta,nama_lengkap,nik_dummy,no_hp,email,alamat_jalan,kelurahan,kecamatan,kota,tanggal_lahir,jenis_kelamin,pendidikan_terakhir,status_disabilitas,jumlah_anak,penghasilan_pribadi_bulanan,nama_usaha,kategori_usaha,produk_utama,lama_usaha_tahun,jumlah_karyawan,omzet_usaha_bulanan,pernah_ikut_pelatihan,kanal_pendaftaran,tanggal_daftar,kehadiran_persen,skor_prates,skor_pascates,tingkat_kepuasan
0,PLT-2025-1309,Ratna Hidayat,99991405899249,8.000426e+10,ratna.hidayat@contoh.example,Jl. Cempaka No. 106 RT 02/RW 10,Sukatani,Tapos,Depok,1989-05-14,P,SMP,Tidak,3,500000,CV Mandiri Jaya,Kuliner,Roti dan kue,12.8,4,2950000,Tidak,Datang langsung,2025-04-04,75,52,67.0,5.0
1,PLT-2025-1512,Bunga Santoso,99992410874363,8.000347e+10,bunga.santoso@contoh.example,Jl. Bougenville No. 80 RT 10/RW 02,Mampang,Pancoran Mas,Depok,1987-10-24,P,D3,Tidak,1,950000,Dapur Cahaya Abadi,Jasa Digital,Servis gawai,6.5,3,7750000,Tidak,Formulir daring,2025-02-02,70,59,75.0,4.9
2,PLT-2025-1360,Endah Ramadhan,99991107987303,8.000266e+10,endah.ramadhan@contoh.example,Jl. Anggrek No. 138 RT 07/RW 08,Limo,Limo,Depok,1998-07-11,P,D3,Tidak,2,1500000,Griya Harum Makmur,Kuliner,Katering rumahan,2.5,0,7550000,Tidak,WhatsApp,2025-03-26,90,51,61.0,4.9


In [3]:
# TODO A1 — kenali kolom yang tersedia sebelum menulis pertanyaan
print("--- DAFTAR KOLOM DAN TIPE DATA ---")
print(df.info())

print("\n--- JUMLAH BARIS DAN KOLOM ---")
print(f"Total Baris: {df.shape[0]} | Total Kolom: {df.shape[1]}")

--- DAFTAR KOLOM DAN TIPE DATA ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 620 entries, 0 to 619
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id_peserta                   620 non-null    object 
 1   nama_lengkap                 620 non-null    object 
 2   nik_dummy                    620 non-null    int64  
 3   no_hp                        611 non-null    float64
 4   email                        620 non-null    object 
 5   alamat_jalan                 620 non-null    object 
 6   kelurahan                    620 non-null    object 
 7   kecamatan                    620 non-null    object 
 8   kota                         620 non-null    object 
 9   tanggal_lahir                620 non-null    object 
 10  jenis_kelamin                620 non-null    object 
 11  pendidikan_terakhir          620 non-null    object 
 12  status_disabilitas           620 non-null  

In [4]:
# TODO A3 — buktikan jawaban nomor 3 dengan kode
kolom_omzet = [col for col in df.columns if 'omzet' in col.lower()]
print("Kolom yang memuat kata 'omzet':", kolom_omzet)
print("Kesimpulan: Sesuai expected, hanya ada satu kolom omzet, yaitu omzet saat mendaftar.")

Kolom yang memuat kata 'omzet': ['omzet_usaha_bulanan']
Kesimpulan: Sesuai expected, hanya ada satu kolom omzet, yaitu omzet saat mendaftar.


In [5]:
from pandas.api.types import CategoricalDtype

# --- TODO B2 — perbaiki satu kolom ordinal yang salah tipe ---
# a. Tentukan urutan jenjang pendidikan
urutan_pendidikan = ['SD', 'SMP', 'SMA/SMK', 'D3', 'S1', 'S2']
kat_pendidikan = CategoricalDtype(categories=urutan_pendidikan, ordered=True)

df['pendidikan_terakhir'] = df['pendidikan_terakhir'].astype(kat_pendidikan)

# b. Tampilkan value_counts() yang sudah urut menurut jenjang
print("--- VALUE COUNTS PENDIDIKAN TERAKHIR ---")
print(df['pendidikan_terakhir'].value_counts())

# c. Cetak median jenjang pendidikan lewat .cat.codes
median_code = df['pendidikan_terakhir'].cat.codes.median()
median_jenjang = urutan_pendidikan[int(round(median_code))]
print(f"\nMedian jenjang pendidikan: {median_jenjang} (Expected: SMA/SMK)\n")


# --- TODO B3 — ubah dua kolom tanggal menjadi bertipe datetime ---
df['tanggal_lahir'] = pd.to_datetime(df['tanggal_lahir'], errors='coerce')
df['tanggal_daftar'] = pd.to_datetime(df['tanggal_daftar'], errors='coerce')

print("--- RENTANG TANGGAL PENDAFTARAN ---")
print("Tanggal pendaftaran paling awal:", df['tanggal_daftar'].min())
print("Tanggal pendaftaran paling akhir:", df['tanggal_daftar'].max())

--- VALUE COUNTS PENDIDIKAN TERAKHIR ---
pendidikan_terakhir
SMA/SMK    281
S1         124
D3         107
SMP         71
SD          24
S2          13
Name: count, dtype: int64

Median jenjang pendidikan: SMA/SMK (Expected: SMA/SMK)

--- RENTANG TANGGAL PENDAFTARAN ---
Tanggal pendaftaran paling awal: 2025-02-01 00:00:00
Tanggal pendaftaran paling akhir: 2025-05-31 00:00:00


In [6]:
from pandas.api.types import CategoricalDtype

# --- TODO B2 — perbaiki satu kolom ordinal yang salah tipe ---
# a. Tentukan urutan jenjang pendidikan
urutan_pendidikan = ['SD', 'SMP', 'SMA/SMK', 'D3', 'S1', 'S2']
kat_pendidikan = CategoricalDtype(categories=urutan_pendidikan, ordered=True)

df['pendidikan_terakhir'] = df['pendidikan_terakhir'].astype(kat_pendidikan)

# b. Tampilkan value_counts() yang sudah urut menurut jenjang
print("--- VALUE COUNTS PENDIDIKAN TERAKHIR ---")
print(df['pendidikan_terakhir'].value_counts())

# c. Cetak median jenjang pendidikan lewat .cat.codes
median_code = df['pendidikan_terakhir'].cat.codes.median()
median_jenjang = urutan_pendidikan[int(round(median_code))]
print(f"\nMedian jenjang pendidikan: {median_jenjang} (Expected: SMA/SMK)\n")


# --- TODO B3 — ubah dua kolom tanggal menjadi bertipe datetime ---
df['tanggal_lahir'] = pd.to_datetime(df['tanggal_lahir'], errors='coerce')
df['tanggal_daftar'] = pd.to_datetime(df['tanggal_daftar'], errors='coerce')

print("--- RENTANG TANGGAL PENDAFTARAN ---")
print("Tanggal pendaftaran paling awal:", df['tanggal_daftar'].min())
print("Tanggal pendaftaran paling akhir:", df['tanggal_daftar'].max())

--- VALUE COUNTS PENDIDIKAN TERAKHIR ---
pendidikan_terakhir
SMA/SMK    281
S1         124
D3         107
SMP         71
SD          24
S2          13
Name: count, dtype: int64

Median jenjang pendidikan: SMA/SMK (Expected: SMA/SMK)

--- RENTANG TANGGAL PENDAFTARAN ---
Tanggal pendaftaran paling awal: 2025-02-01 00:00:00
Tanggal pendaftaran paling akhir: 2025-05-31 00:00:00


In [7]:
# --- TODO D1 — susun tabel penandaan Anda sendiri ---
PENANDAAN = {
    'id_peserta': ('pengenal langsung', 'hash'),
    'nama_lengkap': ('pengenal langsung', 'hapus'),
    'nik_dummy': ('pengenal langsung', 'hapus'),
    'no_hp': ('pengenal langsung', 'hapus'),
    'email': ('pengenal langsung', 'hapus'),
    'alamat_jalan': ('pengenal langsung', 'hapus'),
    'tanggal_lahir': ('quasi-identifier', 'generalisasi'),
    'kelurahan': ('quasi-identifier', 'generalisasi'),
    'kecamatan': ('quasi-identifier', 'pertahankan'),
    'jenis_kelamin': ('quasi-identifier', 'pertahankan'),
    'pendidikan_terakhir': ('quasi-identifier', 'pertahankan'),
    'jumlah_anak': ('data spesifik', 'generalisasi'),
    'omzet_usaha_bulanan': ('data spesifik', 'pertahankan'),
    'penghasilan_pribadi_bulanan': ('data spesifik', 'pertahankan')
}

tanda_data = []
for col in df.columns:
    kategori, tindakan = PENANDAAN.get(col, ('bukan data pribadi', 'pertahankan'))
    tanda_data.append({
        'nama_kolom': col,
        'kategori': kategori,
        'tindakan': tindakan
    })

tanda = pd.DataFrame(tanda_data)

print("--- JUMLAH KOLOM PER KATEGORI DATA PRIBADI ---")
print(tanda['kategori'].value_counts())

display(tanda)

--- JUMLAH KOLOM PER KATEGORI DATA PRIBADI ---
kategori
bukan data pribadi    14
pengenal langsung      6
quasi-identifier       5
data spesifik          3
Name: count, dtype: int64


,nama_kolom,kategori,tindakan
0,id_peserta,pengenal langsung,hash
1,nama_lengkap,pengenal langsung,hapus
2,nik_dummy,pengenal langsung,hapus
3,no_hp,pengenal langsung,hapus
4,email,pengenal langsung,hapus
5,alamat_jalan,pengenal langsung,hapus
6,kelurahan,quasi-identifier,generalisasi
7,kecamatan,quasi-identifier,pertahankan
8,kota,bukan data pribadi,pertahankan
9,tanggal_lahir,quasi-identifier,generalisasi


In [9]:
# --- TODO E1 — HAPUS ---
kolom_hapus = tanda[tanda['tindakan'] == 'hapus']['nama_kolom'].tolist()

aman = df.drop(columns=kolom_hapus, errors='ignore')
print(f"Jumlah kolom yang dibuang: {len(kolom_hapus)}")
print(f"Jumlah kolom yang tersisa di DataFrame 'aman': {aman.shape[1]}")


# --- TODO E2 — HASH ---
GARAM = 'ganti-dengan-kalimat-rahasia-tim'

def samarkan(nilai, garam):
    if pd.isna(nilai):
        return np.nan
    gabung = str(garam) + str(nilai)
    return hashlib.sha256(gabung.encode('utf-8')).hexdigest()[:12]

kolom_hash = tanda[tanda['tindakan'] == 'hash']['nama_kolom'].tolist()
for col in kolom_hash:
    if col in aman.columns:
        aman[col] = aman[col].apply(lambda x: samarkan(x, GARAM))

print("\n--- Kolom yang berhasil di-hash ---")
print(kolom_hash)


# --- TODO E3 — Buktikan hash bersifat konsisten dan tidak terbaca ---
val_contoh = "P001"
h1 = samarkan(val_contoh, GARAM)
h2 = samarkan(val_contoh, GARAM)
print(f"\n[E3a] Hash 1: {h1} | Hash 2: {h2} -> Identik? {h1 == h2}")

h3 = samarkan("P002", GARAM)
print(f"[E3b] Hash 'P002': {h3} -> Berbeda dengan 'P001'? {h1 != h3}")


# --- TODO E4 — GENERALISASI ---
ACUAN = pd.Timestamp('2025-09-01')

# a. Hitung usia peserta
aman['usia'] = (ACUAN - aman['tanggal_lahir']).dt.days // 365

# b. Buat kolom 'kelompok_umur'
bins = [-1, 24, 34, 44, 54, 150]
labels = ['<25', '25-34', '35-44', '45-54', '55+']
aman['kelompok_umur'] = pd.cut(aman['usia'], bins=bins, labels=labels)

# c. Buang kolom tanggal_lahir dan kelurahan dari 'aman'
aman = aman.drop(columns=['tanggal_lahir', 'kelurahan'], errors='ignore')

# d. Rapikan kolom kecamatan (jadikan Title Case)
if 'kecamatan' in aman.columns:
    aman['kecamatan'] = aman['kecamatan'].astype(str).str.title()

print("\n--- Hasil Generalisasi Kelompok Umur ---")
print(aman['kelompok_umur'].value_counts())
print(f"\nJumlah nilai unik kecamatan: {aman['kecamatan'].nunique()}")

Jumlah kolom yang dibuang: 5
Jumlah kolom yang tersisa di DataFrame 'aman': 23

--- Kolom yang berhasil di-hash ---
['id_peserta']

[E3a] Hash 1: e6bbf20b941a | Hash 2: e6bbf20b941a -> Identik? True
[E3b] Hash 'P002': b187e17ee106 -> Berbeda dengan 'P001'? True

--- Hasil Generalisasi Kelompok Umur ---
kelompok_umur
35-44    266
25-34    153
45-54    109
<25       53
55+       39
Name: count, dtype: int64

Jumlah nilai unik kecamatan: 10


In [10]:
# --- TODO F1 — tulis fungsi cek_k(data, kolom) ---
def cek_k(data, kolom):
    grup = data.groupby(kolom).size()
    return pd.Series({
        'k_terkecil': grup.min(),
        'grup_1_orang': (grup == 1).sum(),
        'total_grup': len(grup)
    })


# --- TODO F2 — ukur risiko SEBELUM penanganan ---
kolom_quasi_raw = ['kelurahan', 'jenis_kelamin', 'pendidikan_terakhir', 'tanggal_lahir']
hasil_sebelum = cek_k(df, kolom_quasi_raw)
print("--- RISIKO SEBELUM PENANGANAN ---")
print(hasil_sebelum)


# --- TODO F3 — ukur risiko SESUDAH generalisasi ---
kolom_quasi_aman = ['kecamatan', 'jenis_kelamin', 'pendidikan_terakhir', 'kelompok_umur']
hasil_sesudah = cek_k(aman, kolom_quasi_aman)
print("\n--- RISIKO SESUDAH GENERALISASI ---")
print(hasil_sesudah)


# --- TODO F4 — TANTANGAN: capai k >= 5 ---
# Melepas kolom 'pendidikan_terakhir' agar kombinasi quasi-identifier lebih longgar dan k >= 5 tercapai
kolom_tantangan = ['kecamatan', 'jenis_kelamin', 'kelompok_umur']
hasil_tantangan = cek_k(aman, kolom_tantangan)
print("\n--- HASIL TANTANGAN (k >= 5) ---")
print(hasil_tantangan)


# --- TODO F5 — sel kecil pada tabel agregat ---
# a. Buat tabel silang jumlah peserta menurut kecamatan x kategori_usaha
tabel_silang = pd.crosstab(aman['kecamatan'], aman['kategori_usaha'])

# b. Hitung berapa sel yang terisi tetapi berisi kurang dari 5 peserta (1 s.d. 4)
sel_kecil = ((tabel_silang > 0) & (tabel_silang < 5)).sum().sum()
print(f"\nJumlah sel dengan nilai antara 1 sampai 4: {sel_kecil}")

# Menampilkan cuplikan tabel silang
display(tabel_silang.head())

--- RISIKO SEBELUM PENANGANAN ---
k_terkecil           0
grup_1_orang       620
total_grup      186576
dtype: int64

--- RISIKO SESUDAH GENERALISASI ---
k_terkecil        0
grup_1_orang    137
total_grup      600
dtype: int64

--- HASIL TANTANGAN (k >= 5) ---
k_terkecil        0
grup_1_orang      7
total_grup      100
dtype: int64

Jumlah sel dengan nilai antara 1 sampai 4: 9


/tmp/ipykernel_1327/4080684167.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grup = data.groupby(kolom).size()
/tmp/ipykernel_1327/4080684167.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grup = data.groupby(kolom).size()
/tmp/ipykernel_1327/4080684167.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grup = data.groupby(kolom).size()


kategori_usaha,Fesyen,Jasa Digital,Kriya,Kuliner,Pertanian
kecamatan,,,,,
Beji,14,10,15,42,6
Bojongsari,7,5,4,23,2
Cimanggis,16,8,15,44,7
Cinere,16,11,8,23,9
Cipayung,4,6,5,10,1


In [11]:
# --- G4 — Penanganan Data Tim dan Cek k-Anonymity ---
import pandas as pd
import numpy as np
import hashlib

# 1. Muat dataset proyek tim Anda (pastikan file sudah ada di folder / path yang benar)
# FILE_TIM = 'data/raw/dataset_proyek_tim.csv'
# df_tim = pd.read_csv(FILE_TIM)

# 2. Contoh penerapan penghapusan kolom identitas / quasi-identifier / hashing
# (Sesuaikan kolom dengan dataset tim Anda)
def jalankan_preproteksi(data):
    df_clean = data.copy()

    # Hapus pengenal langsung
    kolom_hapus_tim = ['nama', 'nik', 'no_telepon', 'email']
    df_clean = df_clean.drop(columns=[col for col in kolom_hapus_tim if col in df_clean.columns], errors='ignore')

    # Generalisasi (contoh: kelompok umur)
    if 'tanggal_lahir' in df_clean.columns:
        df_clean['tanggal_lahir'] = pd.to_datetime(df_clean['tanggal_lahir'], errors='coerce')
        acuan = pd.Timestamp('2025-09-01')
        df_clean['usia'] = (acuan - df_clean['tanggal_lahir']).dt.days // 365
        bins = [-1, 24, 34, 44, 54, 150]
        labels = ['<25', '25-34', '35-44', '45-54', '55+']
        df_clean['kelompok_umur'] = pd.cut(df_clean['usia'], bins=bins, labels=labels)
        df_clean = df_clean.drop(columns=['tanggal_lahir', 'usia'], errors='ignore')

    return df_clean

# df_aman_tim = jalankan_preproteksi(df_tim)

# 3. Fungsi cek k untuk data tim
def cek_k_tim(data, kolom_quasi):
    grup = data.groupby(kolom_quasi).size()
    return pd.Series({
        'k_terkecil': grup.min(),
        'grup_1_orang': (grup == 1).sum(),
        'total_grup': len(grup)
    })

# Contoh uji k pada data tim (sesuaikan kolom quasi-identifier tim Anda)
# kolom_q_tim = ['kecamatan', 'jenis_kelamin', 'pendidikan_terakhir', 'kelompok_umur']
# print(cek_k_tim(df_aman_tim, kolom_q_tim))

# 4. Simpan hasil ke folder processed (JANGAN unggah data raw ke GitHub!)
# df_aman_tim.to_csv('data/processed/data_umkm_aman.csv', index=False)
print("Kerangka kode G4 siap disesuaikan dengan dataset tim Anda!")

Kerangka kode G4 siap disesuaikan dengan dataset tim Anda!
